# Homework 2: 影像分類，使用 PyTorch

## 下載 CIFAR-10 資料集

In [ ]:
# 使用 wget 指令下載資料集
# 如果下載很慢，請至 E-Learning 下載

!wget https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz

In [ ]:
# 使用 tar -xvf 進行解壓縮
# 可參考 https://note.drx.tw/2008/04/command.html
!tar -xvf cifar-10-python.tar.gz

In [ ]:
# !md5sum cifar-10-python.tar.gz
# 理論上應該是 c58f30108f718f92721af3b95e74349a

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple
from PIL import Image
import torch
import torchvision
import os
from sklearn.metrics import accuracy_score

In [ ]:
# 觀察 CIFAR-10 的類別

with open("cifar-10-batches-py/batches.meta", "rb") as f:
    label_names = pickle.load(f, encoding='bytes')[b'label_names']
    label_names = [label.decode("utf-8") for label in label_names]

print(label_names)

## 資料前處理
- 我們再來會用到的檔案是 `data_batch_n` (n=1~5), `test_batch` 等資料
- 因此，我們需要先觀察一下它們的內容


In [ ]:
def unpickle(file: str) -> dict:
    with open(file, 'rb') as f:
        dict = pickle.load(f, encoding='bytes')
    return dict

In [ ]:
tmp = unpickle("cifar-10-batches-py/data_batch_1")
print(type(tmp))
print(tmp.keys())

In [ ]:
print(tmp[b'data'].shape)
print(tmp[b'labels'])
print(type(tmp[b'labels']))
print()
print(tmp[b'data'][0].shape)
print(tmp[b'data'][0])
print(type(tmp[b'data']))

### 將影像儲存於對應的類別資料夾

In [ ]:
def save_images_from_batch(
    batch_data: np.ndarray,
    batch_labels: list,
    batch_idx: int,
    split: str,
    output_dir: str,
):
    for i in range(len(batch_data)):
        img_array = batch_data[i] # 每組 batch 的 shape: 張數 x 3096
        label = batch_labels[i]

        img = # TODO1: 將 img_array 轉成 3x32x32，再轉成 (Height, Width, Channel) 格式
        img = Image.fromarray(img) # 將數值資料轉為影像資料

        # 建立類別資料夾，讓影像可以存到各自的類別資料夾之中。
        class_dir = os.path.join(output_dir, split, str(label))
        os.makedirs(class_dir, exist_ok=True)
        # 因為這個 for loop 一次只會取一筆資料，因此需要 exist_ok=True
        # training 資料夾階層可以是 cifar10/train/`class_id`/train_n_0000.png
        # test 資料夾階層可以是 cifar10/test/`class_id`/test_n_0000.png

        img_filename = os.path.join(class_dir, f"{split}_{batch_idx}_{i:04d}.png")
        img.save(img_filename)

In [ ]:
def convert_raw_to_images(
    cifar_dir: str = "cifar-10-batches-py",
    output_dir: str = "cifar10",
):
    os.makedirs(output_dir, exist_ok=True)

    # 處理訓練集
    for i in range(1, 6):
        batch = unpickle(os.path.join(cifar_dir, f"data_batch_{i}"))
        data = batch[b'data']
        labels = batch[b'labels']
        save_images_from_batch(
            batch_data=data,
            batch_labels=labels,
            batch_idx=i,
            split="train",
            output_dir=output_dir,
        )

    # 處理測試集
    test_batch = unpickle(os.path.join(cifar_dir, "test_batch"))
    save_images_from_batch(
        batch_data=test_batch[b'data'],
        batch_labels=test_batch[b'labels'],
        batch_idx=0,
        split="test",
        output_dir=output_dir,
    )

In [ ]:
# 執行資料前處理

convert_raw_to_images()

處理後的資料夾階層將如下所示
```bash
cifar10/
├── train/
│   ├── 0/
│   │   ├── train_0_0001.png
│   │   ├── train_0_0002.png
│   │   └── ...
│   ├── 1/
│   ├── 2/
│   ├── 3/
│   ├── 4/
│   ├── 5/
│   ├── 6/
│   ├── 7/
│   ├── 8/
│   └── 9/
│
└── test/
    ├── 0/
    ├── 1/
    ├── ...
    └── 9/
```

## 建立 PyTorch Dataset

```python
# 繼承 torch.utils.data.Dataset
class CIFAR10ImageFolder(torch.utils.data.Dataset):
    def __init__(self, root_dir: str, transform=None):
        """
        root_dir: 資料夾路徑，例如 'cifar10/train'
        transform: torchvision.transforms 用來轉換 PIL 圖片
        """
    return

    def __getitem__(self, idx: int):
        return

    def __len__(self):
        return
```

In [ ]:
class CIFAR10ImageFolder(torch.utils.data.Dataset):
    def __init__(self, root_dir: str, transform=None):
        """
        root_dir: 資料夾路徑，例如 'cifar10/train'
        transform: torchvision.transforms 用來轉換 PIL 圖片
        """
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []

        # 列出 `root_dir` 底下有哪些類別資料夾，並由小至大排序
        class_names = sorted(os.listdir(root_dir))
        for idx, class_name in enumerate(class_names):
            class_folder = os.path.join(root_dir, class_name)
            if not os.path.isdir(class_folder):
                continue

            for filename in os.listdir(class_folder):
                if filename.endswith(('.png', '.jpg')):
                    # 將所有的檔案路徑都收集到 `self.image_paths`
                    self.image_paths.append(os.path.join(class_folder, filename))
                    # 將所有的 label_id 都收集到 `self.labels`
                    self.labels.append(idx)

    def __getitem__(self, idx: int):
        img_path = self.image_paths[idx] # 取得 idx 所對應的檔案路徑（重要！！）
        label = self.labels[idx]
        image = Image.open(img_path).convert('RGB')  # 讀成 RGB 模式

        if self.transform:
            image = self.transform(image)

        return image, label # 回傳影像資料與標籤

    def __len__(self):
        return len(self.image_paths)

In [ ]:
# 定義資料數值轉換
transform = torchvision.transforms.Compose([
    # transforms.RandomHorizontalFlip(),
    torchvision.transforms.ToTensor(),  # (HWC, uint8) → (CHW, float32 [0,1])
    torchvision.transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

In [ ]:
# 建立 Dataset
train_dataset = CIFAR10ImageFolder(root_dir='cifar10/train', transform=transform)
test_dataset = CIFAR10ImageFolder(root_dir='cifar10/test', transform=transform)

In [ ]:
train_labels = [example[1] for example in train_dataset]
test_labels = [example[1] for example in test_dataset]

In [ ]:
import collections
print(collections.Counter(train_labels))
print(collections.Counter(test_labels))

In [ ]:
# 從 train_dataset 切出 validation_dataset

from torch.utils.data import random_split

validation_ratio = 0.2
total_len = len(train_dataset)
val_size = int(validation_ratio * total_len)
train_size = total_len - val_size

train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

## 超參數

In [ ]:
LR = 0.001
train_batch_size = 64
test_batch_size = 128
num_epochs = 10

## 建立 DataLoader

In [ ]:
# TODO2: 建立 DataLoader (train / valid / test)
train_loader = torch.utils.data.DataLoader(
    # Write your code here
)
val_loader = torch.utils.data.DataLoader(
    # Write your code here
)
test_loader = torch.utils.data.DataLoader(
    # Write your code here
)

# 簡單驗證一個 batch
for images, labels in test_loader:
    print("Image shape:", images.shape)  # [B, 3, 32, 32]
    print("Label shape:", labels.shape)  # [B]
    break

In [ ]:
# 驗證一個 example
example_img, example_label = train_dataset[0]

example = example_img.numpy() # shape: (3, 32, 32)，因為要印出影像，所以需要先轉回 numpy
example = example.transpose(1, 2, 0) # shape: (32, 32, 3)，把 channels 放在最後面

plt.imshow(Image.fromarray((example*255).astype('uint8')))

In [ ]:
# 你也可以單純指定一張影像的位置，來印出它
image = Image.open('cifar10/train/0/train_1_0029.png')
plt.imshow(image)

## 建立模型
- 影像在進行每層 convolution 後的 feature map 大小公式：
$$
\mathrm{output\_size} = \frac{\mathrm{input\_size} - \mathrm{kernel\_size} + 2*\mathrm{padding}}{\mathrm{stride}} + 1
$$

In [ ]:
class SimpleCNN(torch.nn.Module):
    def __init__(self, num_classes=10, fc_hidden_size=256, dropout_ratio=0.5):
        super().__init__()

        self.conv1 = torch.nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = torch.nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = torch.nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.pool = torch.nn.MaxPool2d(kernel_size=2, stride=2) # 2x2 pooling

        self.fc1 = torch.nn.Linear(128 * 4 * 4, fc_hidden_size)
        self.fc2 = torch.nn.Linear(fc_hidden_size, num_classes)
        self.relu = torch.nn.ReLU() # 非線性轉換

        self.dropout = torch.nn.Dropout(dropout_ratio)

    def forward(self, x):
        # TODO3: 加入 conv1 layer
        # Write your code here
        x = self.pool(x)
        # conv1: output_size = (32 - 3 + 2 * 1) / 1 + 1 = 32
        # pooling: output_size = (32 - 3 + 2 * 0) / 2 + 1 = 15.5 (進位為 16)

        # TODO3: 加入 conv2 layer
        # Write your code here
        x = self.pool(x)
        # conv2: output_size = (16 - 3 + 2 * 1) / 1 + 1 = 16
        # pooling: output_size = (16 - 3 + 2 * 0) / 2 + 1 = 7.5 (進位為 8)

        # TODO3: 加入 conv3 layer
        # Write your code here
        x = self.pool(x)   # shape: (B, 128, 4, 4)
        # conv3: output_size = (8 - 3 + 2 * 1) / 1 + 1 = 8
        # pooling: output_size = (8 - 3 + 2 * 0) / 2 + 1 = 3.5 (進位為 4)

        x = x.view(x.size(0), -1)  # Flatten, shape: (B, 128*4*4)
        x = # TODO3: 加入 fc1 layer
        x = # TODO3: 加入 fc2 layer
        return x

## 訓練模型

In [ ]:
# 設定設備
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
# 初始化模型、損失函數、optimizer
model = SimpleCNN().to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

In [ ]:
train_losses, val_losses = [], []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    # TODO4: 請實作 training 流程

    # 驗證模型
    model.eval()
    val_loss = 0.0
    # TODO5: 請實作 validation 流程

    print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {train_losses[-1]:.4f}, '
      f'Val Loss: {val_losses[-1]:.4f}')

# 繪製 loss 的歷史記錄圖
plt.plot(range(1, num_epochs+1), train_losses, label='Train Loss')
plt.plot(range(1, num_epochs+1), val_losses, label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
model.eval()
predictions = []
test_loss = 0.0 # 在 test 的時候其實不用算 loss，因為通常我們沒有 test set 的標準答案
test_losses = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        tmp_pred = outputs.argmax(dim=1) # tmp_pred 的型態是 tensor
        predictions.extend(tmp_pred.tolist()) # 如果用 append 的話會變成 [[1, 2, 3], [4, 5, 6]]

        loss = criterion(outputs, labels)
        test_loss += loss.item()
test_losses.append(test_loss / len(test_loader))

In [ ]:
# test_labels = [example[1] for example in test_dataset]

In [ ]:
accuracy_score(test_labels, predictions)